# 🔬 informed_flow — diagnostics: is a candidate worth trading?

[`informed_flow.ipynb`](./informed_flow.ipynb) optimizes and saves the admissible Pareto front as
candidates. This notebook takes **any set of those candidates** — plus, if you like, what
`config.yaml` ships today — and puts them through every check that decides whether a parameter
set goes to production, **side by side**.

**How to use it.** In §0 set `CANDIDATES` (`"pick"`, trial numbers from the saved front, or
`"config"`) and `FOCUS` (the one candidate the per-name figures draw), then Run All. The checks
run for every candidate; the name-level figures (§2b, §2c, §5c) for `FOCUS` only.

| § | Question | Candidates |
|---|---|---|
| 2 Understand | Which sleeves carry the ranking? Why is a name in or out of the book? | all / FOCUS |
| 3 Stress in-sample | Costs and fill timing, year by year, luck, parameter stability, random entries | all |
| 4 Walk-forward | Does *optimize, then trade* survive unseen data? How did each candidate fare per period? | procedure / all |
| 5 Holdout | The untouched 2025+ data, opened once | all / FOCUS |
| 6 Verdict | One pass/fail table | all |
| 7 Promote | Write a passing candidate into `config.yaml` | one |

**The gates** (thresholds in §0):

| Gate | Pass when |
|---|---|
| Beats buy-and-hold, train | excess > 0 and information ratio > 0 |
| Beats buy-and-hold, holdout | excess > 0 (scored warm) |
| Walk-forward | ≥ 3 of 4 periods positive out of sample and mean > 0 — tests the procedure, so the same for every candidate |
| Selection, not market exposure | holdout alpha after beta > 0 |
| Risk | holdout volatility ≤ buy-and-hold's — a warning unless made hard |
| Not the luckiest of N trials | deflated Sharpe > 0.95 |
| The signal is real | beats the 95th percentile of random-entry books |
| Robust to execution | excess under pessimistic fills — reported; set a threshold to gate on it |
| Not a narrow peak | no parameter-stability `spike` |

Every holdout and walk-forward window is scored **warm** (`score_from`): the strategy runs on
the whole history and only the window's bars count, as they would live. **Do not tune against
§4–§5**: a candidate chosen because it won the holdout has turned the holdout into training data.

## 0 · Choose the candidates

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd
import yaml
from scipy import stats

from trading_simple import backtest as bt
from trading_simple import features, plots, strategies
from trading_simple import optimize as opt
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

# --- what to diagnose ------------------------------------------------------------------------
CANDIDATES = ["pick"]      # "pick", trial numbers from the saved front (e.g. 1191), "config"
FOCUS = None               # the candidate the per-name figures draw (None = the first one)
STUDY = None               # None = config.yaml's optuna.study

# --- the gates (§6), and effort ---------------------------------------------------------------
GATES = {
    "wf_min_positive_periods": 3,   # of 4 walk-forward periods beating buy-and-hold OOS
    "dsr_min": 0.95,                # deflated Sharpe at n_eff = every scored trial
    "random_percentile": 95,        # must beat this percentile of random-entry books
    "vol_is_hard": False,           # False: holdout volatility above buy-and-hold only WARNs
    "pessimistic_min_excess": None,  # e.g. 0.0 to require excess > 0 under worst-side fills
}
HOLDOUT_START = "2025-01-01"
WF_TRIALS = 1000          # trials per walk-forward study (journal-backed: tops up on re-run)
N_RANDOM_SIMS = 200       # random-entry books per candidate
COST_MULTIPLES = (1.0, 2.0, 3.0)

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("informed_flow")
d = strat.data
STUDY = STUDY or strat.optuna["study"]
SPACE = strat.optuna["space"]

saved = yaml.safe_load(opt.study_path(strat, STUDY).with_suffix(".candidates.yaml").read_text())
by_trial = {row["trial"]: row for row in saved["front"]}
FLOOR = saved["pick_min_trades_per_year"]


def resolve(name):
    """A candidate name -> (label, searched params, saved TRAIN metrics or {})."""
    if name == "config":
        return "config", {k: strat.params[k] for k in SPACE}, {}
    trial = saved["pick"] if name == "pick" else int(name)
    if trial not in by_trial:
        raise KeyError(f"trial {trial} is not on the saved front of {STUDY}; "
                       f"choose from {sorted(by_trial)}")
    row = by_trial[trial]
    label = f"pick #{trial}" if trial == saved["pick"] else f"#{trial}"
    return label, row["params"], {k: v for k, v in row.items() if k not in ("trial", "params")}


resolved = [resolve(c) for c in CANDIDATES]
cands = {label: params for label, params, _ in resolved}
FOCUS = resolve(FOCUS)[0] if FOCUS is not None else next(iter(cands))
if FOCUS not in cands:
    raise KeyError(f"FOCUS {FOCUS!r} must be one of CANDIDATES {list(cands)}")
print(f"study {STUDY} · candidates saved {saved['saved']} · {len(by_trial)} on the saved front "
      f"· pick #{saved['pick']} · FOCUS = {FOCUS}")

params_table = pd.DataFrame(cands)
params_table.insert(0, "config.yaml", [strat.params[k] for k in params_table.index])
display(params_table.style.format(lambda v: f"{v:g}" if isinstance(v, float) else str(v)).apply(
    lambda r: ["" if c == "config.yaml" or r[c] == r["config.yaml"] else "background: #fde3c0"
               for c in r.index], axis=1))
print("shaded = differs from what config.yaml ships")
if trained := {label: m for label, _, m in resolved if m}:
    print("\nsaved TRAIN metrics (informed_flow.ipynb §7):")
    display(pd.DataFrame(trained).style.format("{:.3f}"))

## 1 · Load the panels

Two loads, deliberately: the **train** panel exactly as the optimizer saw it, and a separate
**full-history** panel for every out-of-sample score. (Slicing train out of one long load would
leak holdout availability into training data: a feature column exists only where the query
window returned rows for that symbol.) `score()` is the one function every later section reads
its numbers from.

In [ ]:
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))


def load(end):
    """One panel from data.start to `end`, alt-data features attached."""
    p = load_panel(engine, syms, d["bar"], d["start"], end,
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
    features.attach_features(p, engine, d["features"], settings=settings)
    return p


def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx],
                   features={k: v.loc[idx] for k, v in (p.features or {}).items()})


def score(params, p, start=None, strategy=None, cfg=None):
    """Every number the gates read, for one parameter set on one window.

    `start` scores WARM: signals on all of `p`, book, benchmark and metrics from `start` on.
    Adds buy-and-hold's CAGR and volatility, beta against it and the alpha left after beta.
    """
    strategy, cfg = strategy or strat, cfg or settings
    pf = bt.run_strategy(strategy, p, params, settings=cfg, score_from=start)
    w = p if start is None else slice_panel(p, start, str(p.close.index[-1].date()))
    out = bt.portfolio_stats(pf)
    fees, slip = bt.panel_costs(cfg.costs, w)
    out |= bt.active_stats(pf, w, fees=fees, slippage=slip, total_return=out["total_return"],
                           fill=strategy.fill)
    bench_ret, bench = bt.benchmark(w, fees=fees, slippage=slip, fill=strategy.fill)
    r = bt.equity_curve(pf)["value"].pct_change().fillna(0.0)
    bars_per_year = len(w.close.index) / out["years"]
    out["bench_cagr"] = bt._cagr(bench_ret, out["years"])
    out["bench_volatility"] = float(bench.std() * np.sqrt(bars_per_year))
    out["beta"] = float(np.cov(r, bench)[0, 1] / bench.var())
    out["alpha"] = float((r.mean() - out["beta"] * bench.mean()) * bars_per_year)
    out["active"] = (r - bench).to_numpy()
    out["pf"] = pf
    return out


PCT = "{:+.1%}"
panel = load(d.get("end"))          # TRAIN — what the study was fitted on
full = load(None)                   # the whole history — every warm out-of-sample score
LAST = str(full.close.index[-1].date())
holdout = slice_panel(full, HOLDOUT_START, LAST)    # charts only — never backtest a slice
print(f"train {panel.close.index[0].date()} → {panel.close.index[-1].date()} "
      f"({len(panel.close)} bars) · holdout {HOLDOUT_START} → {LAST} ({len(holdout.close)} bars)")

## 2 · Understand the candidates

### 2a · Does the ranking pick winners? (train)

A backtest mixes three things: whether the ranking picks winners, how the book is built (slots,
exits, costs), and simply being in a rising market. This table measures **only the first**,
without trading anything: do the names ranked high go on to beat the names ranked low? Each day's
market move is removed first, so what is left is stock picking alone. One row per ingredient
(sleeve), plus the combined ranking the book actually trades; one table per candidate.

For each horizon — **1, 3 and 6 months ahead**:

- **IC** — does a higher rank lead to a better return? Above 0 = yes, a little; below 0 = the
  ingredient points the wrong way. Real signals here are small: +0.03 is already respectable.
- **t** — how sure we can be it is not luck (adjusted for overlapping windows). Below 2: can't
  tell it apart from luck.
- **spread** — the top 10% of names minus the bottom 10% over that horizon (not per year).

**How to read across the horizons.** A real edge holds or grows from 1 to 6 months; one that
fades after a month was short-lived. The book holds names for roughly one to a few months, so the
1- and 3-month columns are the ones it actually harvests. A sleeve with a negative spread is not
neutral — its weight pulls the ranking the wrong way (the "insider off" ablation in §4b tests
that on unseen data). A diagnostic, not a gate: a thin edge here is a reason to trust §3e and the
alpha gate more than the backtest's headline return.

In [ ]:
HORIZONS = (21, 63, 126)          # ~1, 3 and 6 months ahead
HORIZON_NAMES = {21: "1 month", 63: "3 months", 126: "6 months"}
mod = strat.signals.__globals__
compose = mod["_compose"]


def signal_quality(score_frame, close):
    """Rank IC (overlap-adjusted t) and decile spread against DEMEANED forward returns."""
    rows, r = [], score_frame.rank(axis=1, pct=True)
    for h in HORIZONS:
        fwd = close.shift(-h) / close - 1.0
        rel = fwd.sub(fwd.mean(axis=1), axis=0)              # the market, removed
        ic = score_frame.corrwith(rel, axis=1, method="spearman").dropna()
        naive_t = ic.mean() / (ic.std() / np.sqrt(len(ic))) if len(ic) and ic.std() else 0.0
        rows.append({"horizon": f"{h}d", "rank_IC": float(ic.mean()),
                     "t": float(naive_t / np.sqrt(h)),       # consecutive ICs overlap by h-1
                     "spread": float(rel.where(r >= 0.9).mean(axis=1).mean()
                                     - rel.where(r <= 0.1).mean(axis=1).mean())})
    return pd.DataFrame(rows).set_index("horizon")


def sleeve_table(p, params, start=None):
    """IC, t and top-minus-bottom spread per sleeve and horizon; `start` scores warm."""
    parts = compose(p, **{**strat.params, **params})
    close = p.close if start is None else p.close.loc[start:]
    rows = {}
    for k in parts["present"] + ["COMPOSITE"]:
        frame = parts["score_pct"] if k == "COMPOSITE" else parts["ranks"][k]
        q = signal_quality(frame.loc[close.index], close)
        rows[k] = {(HORIZON_NAMES[h], name): q.loc[f"{h}d", col] for h in HORIZONS
                   for name, col in (("IC", "rank_IC"), ("t", "t"), ("spread", "spread"))}
    return pd.DataFrame(rows).T


def show_ic(tables, window):
    """One table per candidate: IC, t and spread at every horizon."""
    for label, table in tables.items():
        print(f"{window} — {label}")
        spread = [c for c in table.columns if c[1] == "spread"]
        t_cols = [c for c in table.columns if c[1] == "t"]
        display(table.style.format("{:+.3f}").format(PCT, subset=spread)
                .format("{:+.2f}", subset=t_cols))


show_ic({label: sleeve_table(panel, p) for label, p in cands.items()}, "TRAIN")

### 2b · Why is a name in or out of the book? (FOCUS)

A name is bought when its **cross-sectional percentile rank** crosses `q_in` and a slot is free,
and sold when the rank falls through `q_out`. `strategy.explain()` returns the audit trail and
shares its whole computation with `signals()`, so this picture cannot drift from what traded.
Three things it makes visible, each of which looks like a bug until you see it:

1. **A name can cross with no event of its own** — the rank is relative, so it rises when its
   peers fall. The `own`/`peer` split attributes every rank change exactly.
2. **A name's own value moves with no new event** — an old insider buy rolling off the back of
   its trailing window moves the score by itself.
3. **A missing signal is a neutral 0.5, not a zero** — hatched bars carry no information.

Hollow circle = qualified, ✕ = qualified but no free slot, ▲ = the actual fill (features lag a
bar and fills are next-open, so the fill trails the crossing); shaded spans are real positions.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple.plots import DIVERGING, SERIES

explain = mod["explain"]
PARAMS = {**strat.params, **cands[FOCUS]}
PAD = 60                       # bars either side of the crossing we zoom on

res_focus = bt.run_strategy(strat, panel, params=cands[FOCUS], settings=settings)
trades = bt.trade_records(res_focus)
# Explain the name the strategy traded most — that is where there is something to see.
SYMBOL = (trades["symbol"].value_counts().idxmax() if len(trades)
          else panel.features["insider"].abs().sum().idxmax())

exp = explain(panel, SYMBOL, **PARAMS)
mine = trades[trades["symbol"] == SYMBOL]

# Zoom on the first entry; fall back to the strongest rank jump if it never traded.
focus = (pd.Timestamp(mine["entry_ts"].iloc[0]) if len(mine)
         else exp["d_score"].abs().idxmax())
i_focus = exp.index.get_indexer([focus], method="nearest")[0]
start, stop = max(i_focus - PAD, 0), i_focus + PAD
# The slow trend SMA has no value until `trend_slow` bars exist: reach past its warm-up so the
# fast/slow crossover in row ④ is actually visible.
stop = max(stop, PARAMS["trend_slow"] + PAD)
win = exp.iloc[start: min(stop, len(exp))]

print(f"{FOCUS} · {SYMBOL}: {len(mine)} trades, zooming {win.index[0].date()} → "
      f"{win.index[-1].date()} ({len(win)} bars)")
print(f"sleeves in the composite: {[c[5:] for c in exp.columns if c.startswith('rank_')]}")
if dropped := set(panel.features) - {c[5:] for c in exp.columns if c.startswith("rank_")}:
    print(f"⚠ dropped (zero weight, or below the coverage floor): {sorted(dropped)}")

In [ ]:
SLEEVES = [c[5:] for c in exp.columns if c.startswith("rank_")]
q_in, q_out = PARAMS["q_in"], PARAMS["q_out"]

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, row_heights=[0.30, 0.22, 0.24, 0.24],
    vertical_spacing=0.05,
    specs=[[{}], [{}], [{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=("① Composite rank vs the q_in / q_out bands",
                    "② Why: each sleeve's weighted contribution (stacked = composite)",
                    "③ The events that moved it, as the strategy saw them",
                    "④ Price regime: close, fast & slow SMAs (the trend sleeve) and RSI"))
fig.update_annotations(font_size=13)

# --- ① the rank the bands are compared against ----------------------------------------------
fig.add_trace(go.Scatter(x=win.index, y=win["score_pct"], name="composite rank",
                         line={"color": SERIES[0], "width": 2},
                         customdata=np.c_[win["own"], win["peer"], win["driver"]],
                         hovertemplate=("rank %{y:.2f}<br>own %{customdata[0]:+.3f}"
                                        " · peer %{customdata[1]:+.3f}"
                                        " → %{customdata[2]}-driven<extra></extra>")),
              row=1, col=1)
cross = win.index[win["qualified"]]
fig.add_trace(go.Scatter(
    x=cross, y=win["score_pct"].reindex(cross), mode="markers", name="qualified",
    marker={"symbol": "circle-open", "size": 13, "color": SERIES[3],
            "line": {"width": 2, "color": SERIES[3]}}), row=1, col=1)
# A "-thin" symbol has no fill: plotly strokes it from marker.line.color.
if (turned_away := win.index[win["slot_blocked"]]).size:
    fig.add_trace(go.Scatter(
        x=turned_away, y=win["score_pct"].reindex(turned_away), mode="markers",
        name="NO SLOT (book full)",
        marker={"symbol": "x-thin", "size": 10, "color": "#c0392b",
                "line": {"width": 2.5, "color": "#c0392b"}}), row=1, col=1)
w0, w1 = win.index[0], win.index[-1]
# vectorbt records one row per LOT: dedupe to the bars the book actually opened on.
fills = pd.DatetimeIndex(sorted(set(pd.to_datetime(mine["entry_ts"])))).intersection(win.index)
fig.add_trace(go.Scatter(
    x=fills, y=win["score_pct"].reindex(fills), mode="markers", name="actual fill",
    marker={"symbol": "triangle-up", "size": 15, "color": SERIES[4],
            "line": {"width": 1.5, "color": "#ffffff"}}), row=1, col=1)
if (blocked := win.index[~win["eligible"]]).size:
    fig.add_trace(go.Scatter(
        x=blocked, y=np.zeros(len(blocked)), mode="markers", name="ineligible (min_events)",
        marker={"symbol": "line-ns-open", "size": 8, "color": "#898781",
                "line": {"width": 1.5, "color": "#898781"}}), row=1, col=1)

# --- ② attribution; hatched = neutral-filled (NO DATA) --------------------------------------
for i, k in enumerate(SLEEVES):
    gap = win[f"filled_{k}"]
    fig.add_trace(go.Bar(
        x=win.index, y=win[f"contrib_{k}"], name=k, marker_color=SERIES[i],
        marker_pattern={"shape": ["/" if g else "" for g in gap],
                        "fgcolor": "#ffffff", "solidity": 0.35},
        customdata=np.c_[win[f"rank_{k}"], gap],
        hovertemplate=(f"<b>{k}</b><br>contribution %{{y:.3f}}<br>rank %{{customdata[0]:.2f}}"
                       "<br>no data (neutral 0.5): %{customdata[1]}<extra></extra>")),
        row=2, col=1)
fig.add_trace(go.Scatter(x=win.index, y=win["composite"], name="composite",
                         line={"color": "#0b0b0b", "width": 1.5}), row=2, col=1)

# --- ③ the raw point-in-time events ---------------------------------------------------------
if "insider_usd" in win:
    v = win["insider_usd"].fillna(0)
    fig.add_trace(go.Bar(x=win.index, y=v, name="insider net $",
                         marker_color=[DIVERGING[2][1] if x > 0 else DIVERGING[0][1] for x in v],
                         hovertemplate="insider $%{y:,.0f}<extra></extra>"), row=3, col=1)
for col, label, shape in [("rating_change", "rating Δ", "triangle-up"),
                          ("ptrev", "price-target rev", "diamond")]:
    if col not in win:
        continue
    ev = win[col].fillna(0)
    ev = ev[ev != 0]
    fig.add_trace(go.Scatter(
        x=ev.index, y=np.zeros(len(ev)), mode="markers", name=label,
        marker={"symbol": [shape if x > 0 else shape.replace("-up", "-down") for x in ev],
                "size": 10, "color": [SERIES[3] if x > 0 else SERIES[5] for x in ev]},
        customdata=ev.to_numpy(),
        hovertemplate=f"{label} %{{customdata:+.3f}}<extra></extra>"), row=3, col=1)
if "days_to_cover" in win:
    fig.add_trace(go.Scatter(x=win.index, y=win["days_to_cover"], name="days-to-cover",
                             line={"color": SERIES[6], "width": 1.5, "shape": "hv"}),
                  row=3, col=1, secondary_y=True)

# --- ④ price regime: close, the two SMAs of the trend sleeve, RSI ----------------------------
_rsi = mod["_rsi"]
tf, ts, rp = PARAMS["trend_fast"], PARAMS["trend_slow"], PARAMS["rsi_period"]
close_w = panel.close[SYMBOL].reindex(win.index)
fast_w = bt.sma(panel.close, tf)[SYMBOL].reindex(win.index)
slow_w = bt.sma(panel.close, ts)[SYMBOL].reindex(win.index)
rsi_w = _rsi(panel.close, rp)[SYMBOL].reindex(win.index)
fig.add_trace(go.Scatter(x=win.index, y=close_w, name="close (reference)",
                         line={"color": "#9b988f", "width": 1.3},
                         hovertemplate="close $%{y:,.2f}<extra></extra>"), row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=fast_w, name=f"fast SMA {tf}",
                         line={"color": SERIES[0], "width": 2.2},
                         hovertemplate=f"fast SMA {tf} $%{{y:,.2f}}<extra></extra>"),
              row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=slow_w, name=f"slow SMA {ts}",
                         line={"color": SERIES[7], "width": 2.2, "dash": "dash"},
                         hovertemplate=f"slow SMA {ts} $%{{y:,.2f}}<extra></extra>"),
              row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=rsi_w, name=f"RSI {rp}",
                         line={"color": SERIES[6], "width": 1.5},
                         hovertemplate="RSI %{y:.0f}<extra></extra>"),
              row=4, col=1, secondary_y=True)
# Guides and shapes go on AFTER the traces: add_hline/add_hrect/add_vrect silently drop a shape
# aimed at a subplot that has no trace yet.
for lvl, tag, pos in [(70, "RSI 70 · overbought", "top right"),
                      (30, "RSI 30 · oversold", "bottom right")]:
    fig.add_hline(y=lvl, line={"color": "#b0ada5", "width": 1, "dash": "dot"},
                  row=4, col=1, secondary_y=True,
                  annotation_text=tag, annotation_position=pos,
                  annotation_font_size=10, annotation_font_color="#898781")
fig.add_hrect(y0=q_in, y1=1.0, fillcolor=SERIES[1], opacity=0.10, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"IN (rank ≥ q_in = {q_in})", annotation_position="top left",
              annotation_font_size=11, annotation_font_color="#898781")
fig.add_hrect(y0=q_out, y1=q_in, fillcolor=SERIES[2], opacity=0.09, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"HOLD band (q_out = {q_out} – {q_in})",
              annotation_position="bottom left",
              annotation_font_size=11, annotation_font_color="#898781")
# REAL positions from the portfolio: union the FIFO lots into the spans the book was long.
lots = mine[["entry_ts", "exit_ts"]].apply(pd.to_datetime).sort_values("entry_ts")
held: list[list[pd.Timestamp]] = []
for entry, exit_ in lots.itertuples(index=False):
    if held and entry <= held[-1][1]:
        held[-1][1] = max(held[-1][1], exit_)
    else:
        held.append([entry, exit_])
for entry, exit_ in held:
    if exit_ >= w0 and entry <= w1:
        fig.add_vrect(x0=max(entry, w0), x1=min(exit_, w1),
                      fillcolor=SERIES[3], opacity=0.08, line_width=0, layer="below")

fig.update_layout(
    title={"text": (f"<b>{SYMBOL}</b> — why it is in or out of the book ({FOCUS})<br>"
                    "<sup>○ qualified · ✕ no slot · ▲ actual fill · shaded span = position held"
                    " · hatched bar = NO DATA, neutral 0.5</sup>"),
           "font": {"size": 15, "color": "#0b0b0b"},
           "x": 0, "xref": "paper", "xanchor": "left",
           "y": 0.99, "yref": "container", "yanchor": "top"},
    template="plotly_white", height=1200, barmode="relative", bargap=0.05,
    font={"family": 'system-ui, -apple-system, "Segoe UI", sans-serif', "size": 12},
    hovermode="x unified",
    legend={"orientation": "h", "x": 0.5, "xanchor": "center",
            "y": 1.06, "yanchor": "bottom", "font": {"size": 11}},
    margin={"l": 70, "r": 70, "t": 270, "b": 50})
fig.update_xaxes(range=[w0, w1])       # pin the zoom: a shape must not drag the autorange
fig.update_yaxes(title_text="rank (0–1)", range=[-0.02, 1.02], row=1, col=1)
fig.update_yaxes(title_text="contribution", row=2, col=1)
fig.update_yaxes(title_text="insider $ / events", row=3, col=1)
fig.update_yaxes(title_text="days-to-cover", showgrid=False, row=3, col=1, secondary_y=True)
fig.update_yaxes(title_text="price ($)", row=4, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], showgrid=False, row=4, col=1,
                 secondary_y=True)
fig.show()

# Every bar this name crossed a band, with the cause attributed. driver == "peer" and
# own_event == False is the counter-intuitive case: the cross-section moved under the name.
zone_prev = exp["zone"].shift(1)
crossings = exp[(exp["zone"] != zone_prev) & zone_prev.notna()].copy()
crossings["move"] = zone_prev[crossings.index] + " → " + crossings["zone"]
cols = ["move", "score_pct", "d_score", "own", "peer", "driver", "own_event", "eligible"]
display(crossings[cols + [f"d_contrib_{k}" for k in SLEEVES]].round(3).tail(12))
peer_driven = crossings[(crossings["driver"] == "peer") & ~crossings["own_event"]]
print(f"{len(crossings)} band crossings for {SYMBOL}; {len(peer_driven)} were pure PEER DRIFT "
      "— the name crossed with no news of its own.")

### 2c · The whole book at once — who is held, and when (FOCUS)

Every symbol's daily state: **held** (a real position), **qualified — no slot** (ranked above
`q_in` but the book was full: the capital constraint, and the answer to "its rank is 0.95, why
do I not own it?"), **band** (inside the `q_out`–`q_in` hysteresis zone), **out**, and
**ineligible** (too few events to be ranked at all). Read it for **churn** (names flickering in
and out pay costs for nothing), **breadth** (the same handful always held means the
cross-sectional thesis is not doing the work) and **pressure** (a wall of no-slot days means the
ranking has more conviction than the book has room for).

In [ ]:
q_in, q_out = PARAMS["q_in"], PARAMS["q_out"]
parts = compose(panel, **PARAMS)
selected = mod["_select"](parts, q_in=q_in, q_out=q_out, max_hold=PARAMS["max_hold"],
                          max_positions=PARAMS["max_positions"],
                          displace_margin=PARAMS["displace_margin"])
score_pct, eligible = parts["score_pct"], parts["eligible"]

# 0 ineligible · 1 out · 2 band · 3 qualified but NO SLOT · 4 held (a NaN rank falls to 1, then 0)
state = np.where(score_pct >= q_in, 3, np.where(score_pct >= q_out, 2, 1))
state = np.where(eligible.to_numpy(), state, 0)
state = pd.DataFrame(state, index=score_pct.index, columns=score_pct.columns)
state = state.mask(selected["slot_blocked"], 3)
for _, t in trades.iterrows():   # truth beats reconstruction
    state.loc[pd.Timestamp(t["entry_ts"]):pd.Timestamp(t["exit_ts"]), t["symbol"]] = 4

order = state.eq(4).sum().sort_values(ascending=False).index   # most-held names first
z = state[order].T
LABELS = ["ineligible", "out", "band", "qualified — NO SLOT", "held"]
COLORS = ["#e8e6df", "#f4f3ef", "#fde3c0", "#e8a49c", SERIES[3]]
bands = [[i / 5, COLORS[j]] for j in range(5) for i in (j, j + 1)]
fig = go.Figure(go.Heatmap(
    z=z.to_numpy(), x=z.columns, y=z.index, colorscale=bands, zmin=0, zmax=5,
    colorbar={"tickvals": [0.5, 1.5, 2.5, 3.5, 4.5], "ticktext": LABELS, "len": 0.6},
    hovertemplate="%{y} · %{x|%Y-%m-%d}<extra></extra>"))
fig.update_layout(
    title=(f"Book state by name ({FOCUS}) — {len(order)} symbols · q_in={q_in}, "
           f"q_out={q_out} · {PARAMS['max_positions']} slots · held spans are real trades"),
    template="plotly_white", height=max(460, 11 * len(order)),
    margin={"l": 70, "r": 20, "t": 70, "b": 40})
fig.update_yaxes(tickfont={"size": 8})
fig.show()
print(f"held: {int(state.eq(4).sum().sum()):,} symbol-days across "
      f"{int(state.eq(4).any().sum())} names · qualified but NO SLOT on "
      f"{selected['slot_blocked'].sum(axis=1).mean():.1f} names an average day · "
      f"{parts['n_ranked'].mean():.0f} of {len(score_pct.columns)} names ranked per day")

## 3 · Stress the candidates in-sample (train)

Everything in this section reads the training window only — the one the study was fitted on —
so it can show a candidate is fragile, never that it is good. First the baseline comparison:
each candidate against equal-weight buy-and-hold of the same names, with the numbers the gates
read (volatility is the annual risk rate the optimizer minimizes; alpha is what is left after
removing the market exposure `beta` explains).

In [ ]:
train = {label: score(p, panel) for label, p in cands.items()}
COLS = {"cagr": PCT, "bench_cagr": PCT, "excess_cagr": PCT, "info_ratio": "{:+.2f}",
        "volatility": "{:.1%}", "bench_volatility": "{:.1%}", "max_drawdown": PCT,
        "beta": "{:.2f}", "alpha": PCT, "trades_per_year": "{:.0f}"}
display(pd.DataFrame({label: {k: s[k] for k in COLS} for label, s in train.items()}).T
        .style.format(COLS))

### 3a · Costs and fill timing

Each candidate at 1×/2×/3× the configured costs, the half-spread per leg (bps) at which its
excess over buy-and-hold reaches zero, and its excess when every buy fills at the bar's high and
every sell at its low (`fill: pessimistic`, benchmark included). Costs are a spread you pay;
fill timing is where in the day you trade, and only the second can erase a daily strategy's
edge. Pessimistic fills are a bound, not a forecast: the truth lies between the two.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})


def excess(params, strategy=strat, cfg=settings):
    """Train excess over buy-and-hold per year."""
    return bt.evaluate(strategy, panel, params, settings=cfg)["excess_cagr"]


def breakeven_bps(params):
    """Half-spread per leg (bps) at which the train excess over buy-and-hold reaches zero."""
    base = settings.costs.spread_bps + settings.costs.slippage_bps
    for extra in np.arange(0, 305, 10.0):
        if excess(params, cfg=with_costs(spread_bps=base + extra, slippage_bps=0.0)) <= 0:
            return base + extra
    return np.inf


c, worst = settings.costs, replace(strat, fill="pessimistic")
costs = {}
for label, p in cands.items():
    row = {f"{m:g}x costs": excess(p, cfg=with_costs(
        spread_bps=c.spread_bps * m, slippage_bps=c.slippage_bps * m,
        commission_per_share=c.commission_per_share * m)) for m in COST_MULTIPLES}
    row["breakeven bps/leg"] = breakeven_bps(p)
    row["pessimistic fills"] = excess(p, strategy=worst)
    costs[label] = row
costs = pd.DataFrame(costs).T
display(costs.style.format(PCT).format("{:.0f}", subset=["breakeven bps/leg"]))

### 3b · Year by year (train)

Each calendar year of the training window on its own, scored warm. An edge carried by one or two
years is a regime bet, not a strategy.

In [ ]:
years = {}
for y in range(pd.Timestamp(d["start"]).year, pd.Timestamp(d["end"]).year + 1):
    sub = slice_panel(panel, d["start"], f"{y}-12-31")        # history + the year
    years[y] = {label: bt.evaluate(strat, sub, p, settings=settings,
                                   score_from=f"{y}-01-01")["excess_cagr"]
                for label, p in cands.items()}
year_table = pd.DataFrame(years).T.rename_axis("excess per year (train)")
display(year_table.style.format(PCT).background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3))

### 3c · Deflated Sharpe — the best of N trials beats a naive threshold by luck

Bailey & López de Prado (2014): the candidate's information ratio against buy-and-hold,
discounted for how many trials the study tried and for the skew and fat tails of its active
returns. It asks whether the edge over buy-and-hold is real rather than the luckiest of N draws.
NSGA-II trials are correlated, so it is shown across a band of effective trial counts; the gate
reads the strictest (every scored trial).

In [ ]:
trial_ir = opt.trials_frame(opt.load_study(strat, STUDY))["info_ratio"].dropna().to_numpy()
N_TRIALS = len(trial_ir)
ANN = np.sqrt(252)


def deflated(score_ann, returns, n_eff):
    """P(the edge over buy-and-hold is genuinely > 0 after `n_eff` tries)."""
    gamma = 0.5772156649
    e_max = np.std(trial_ir, ddof=1) * ((1 - gamma) * stats.norm.ppf(1 - 1.0 / n_eff)
                                        + gamma * stats.norm.ppf(1 - 1.0 / (n_eff * np.e)))
    sr, sr0 = score_ann / ANN, e_max / ANN
    sk, ku = stats.skew(returns), stats.kurtosis(returns, fisher=False)
    denom = np.sqrt(1 - sk * sr + (ku - 1) / 4 * sr ** 2)
    return float(stats.norm.cdf((sr - sr0) * np.sqrt(len(returns) - 1) / denom))


N_EFFS = sorted({max(2, N_TRIALS // 4), max(2, N_TRIALS // 2), N_TRIALS})
dsr = pd.DataFrame({label: {f"n_eff={n}": deflated(s["info_ratio"], s["active"], n)
                            for n in N_EFFS}
                    for label, s in train.items()}).T
display(dsr.style.format("{:.3f}"))
print(f"{N_TRIALS} scored trials in {STUDY}; want > {GATES['dsr_min']} at n_eff={N_TRIALS}.")

### 3d · Parameter stability — plateau or spike?

Each searched knob moved one at a time by ±5% and ±10% of its own searched span (log-aware,
snapped to values the study could have suggested), re-scored on train. The summary counts the
verdicts per candidate; the `FOCUS` candidate's full table and tornado follow.

| verdict | meaning | action |
|---|---|---|
| `inert` | the neighbourhood scores the same | pin it; spend trials elsewhere (a local claim) |
| `plateau` | moves cost something, nothing breaks | fine |
| `cliff (low/high)` | one side loses half the edge in one step | take `suggest`, one step clear of the edge |
| `spike` | both sides collapse | **do not ship** — prefer a flatter front row |
| `off-center` | a neighbour beats it in sample | not an instruction: refer it to the walk-forward |
| `decisive` | (a categorical) the other choice loses the edge | keep it |

In [ ]:
stability = {}
for label, p in cands.items():
    star = opt.sensitivity(strat, panel, p, settings=settings)
    verdicts = opt.stability_report(star, min_trades=FLOOR * train[label]["years"])
    center = float(star.loc[star["perturbed"] == "", "excess_cagr"].iloc[0])
    near = star.loc[star["perturbed"] != "", "excess_cagr"]
    stability[label] = {
        "neighbours": len(near), "beat buy & hold": int((near > 0).sum()),
        "median keeps": near.median() / center if center > 0 else np.nan,
        "spikes": int((verdicts["verdict"] == "spike").sum()),
        "cliffs": int(verdicts["verdict"].str.startswith("cliff").sum()),
        "inert": int((verdicts["verdict"] == "inert").sum())}
    if label == FOCUS:
        focus_star, focus_verdicts = star, verdicts
display(pd.DataFrame(stability).T.style.format("{:.0f}").format("{:.0%}", subset=["median keeps"]))
print(f"\n{FOCUS}, knob by knob:")
display(focus_verdicts)
plots.plot_sensitivity(focus_star).show()

### 3e · Random-entry baseline — is it the signal, or the construction?

The same book — same slots, hysteresis, displacement, risk overlay and costs — driven by
**random** rankings instead of the real one. If random books score like the candidate, its edge
came from the construction (concentration, staying invested), not from the ranking. The
candidate should beat the chosen percentile of the random distribution.

In [ ]:
fees_t, slip_t = bt.panel_costs(settings.costs, panel)
bench_cagr_t = bt._cagr(bt.benchmark(panel, fees=fees_t, slippage=slip_t, fill=strat.fill)[0],
                        bt._years(panel.close.index))


def book_excess(parts, p, ranking):
    """Slot machinery + risk overlay on any ranking frame -> its train excess CAGR."""
    held = mod["_select"](dict(parts, score_pct=ranking), q_in=p["q_in"], q_out=p["q_out"],
                          max_hold=p["max_hold"], max_positions=p["max_positions"],
                          displace_margin=p["displace_margin"])["held"]
    weights = mod["_weights"](held, p["max_positions"])
    weights = weights.mul(mod["_exposure"](panel, weights, **p), axis=0)
    pf = bt.run(panel, {"weights": weights}, fees=fees_t, slippage=slip_t,
                init_cash=settings.backtest.init_cash, fill=strat.fill)
    return bt.portfolio_stats(pf)["cagr"] - bench_cagr_t


rng = np.random.default_rng(7)
PCTL = GATES["random_percentile"]
randoms = {}
for label, params in cands.items():
    p = {**strat.params, **params}
    parts = compose(panel, **p)
    real_rank = parts["score_pct"]
    real = book_excess(parts, p, real_rank)        # the same machinery on the REAL ranking
    sims = np.array([
        book_excess(parts, p, pd.DataFrame(rng.random(real_rank.shape), index=real_rank.index,
                                           columns=real_rank.columns)
                    .where(parts["eligible"]).rank(axis=1, pct=True))
        for _ in range(N_RANDOM_SIMS)])
    randoms[label] = {"real ranking": real, "random mean": sims.mean(),
                      f"random p{PCTL}": np.percentile(sims, PCTL),
                      "z": (real - sims.mean()) / sims.std() if sims.std() else np.nan}
randoms = pd.DataFrame(randoms).T
display(randoms.style.format(PCT).format("{:+.1f}", subset=["z"]))
print("book_excess leaves out the retrim_k concentration cap for speed, on BOTH sides, so these "
      "numbers\ndiffer slightly from §3's; the comparison between them is like for like.")

## 4 · Out of sample — the walk-forward

A walk-forward tests the **procedure**, not a parameter set: for each of four half-years it
re-runs the full study on the data before it — same space, sampler and picking rule as the
optimization notebook — then scores that period's pick on the unseen half-year that follows,
warm, with the history carried in. *If I optimize today and trade tomorrow, does what I pick keep
working?* That verdict is the same for every candidate.

Two more views ride along:

- **Each candidate on each period** — how the candidates themselves would have done in those
  half-years. Their study saw these months, so this is **in-sample** for them: it shows whether a
  candidate's edge is spread across periods or sits in one, never a pass/fail.
- **§4b ablations** — each period's own pick re-scored with the recency sleeve at 1.0 and off,
  the insider sleeve off, and under each risk method: do those choices earn their place out of
  sample?

Four half-years is a thin sample (a paired t-test over four folds barely detects anything):
read the spread across periods, not just the mean.

In [ ]:
FOLDS = [(d["start"], "2022-12-31", "2023-01-01", "2023-06-30"),
         (d["start"], "2023-06-30", "2023-07-01", "2023-12-31"),
         (d["start"], "2023-12-31", "2024-01-01", "2024-06-30"),
         (d["start"], "2024-06-30", "2024-07-01", "2024-12-31")]
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}


def as_params(row) -> dict:
    """A front row's searched parameters, cast back to the types signals() expects."""
    return {k: (row[k] if isinstance(row[k], str) else int(round(float(row[k])))
                if k in SPACE_INT else round(float(row[k]), 10)) for k in SPACE}


wf, per_candidate = [], {}
for i, (tr0, tr1, te0, te1) in enumerate(FOLDS, 1):
    name = f"{STUDY}-wf{i}"
    try:
        done = len(opt.load_study(strat, name).trials)
    except KeyError:
        done = 0
    fold_study = opt.run_study(strat, slice_panel(panel, tr0, tr1), settings=settings,
                               study_name=name, n_trials=max(0, WF_TRIALS - done))
    fold_front = opt.pareto_frame(fold_study)
    pool = fold_front[fold_front["trades_per_year"] >= FLOOR]
    row = opt.pick_from_front(pool if len(pool) else fold_front)
    window = slice_panel(panel, tr0, te1)       # history + the test half-year, scored warm
    oos = bt.evaluate(strat, window, as_params(row), settings=settings, score_from=te0)
    period = f"{te0[:7]}..{te1[:7]}"
    wf.append({"period": period, "picked": as_params(row), "window": window, "te0": te0,
               "IS excess": float(row["excess_cagr"]), "OOS excess": oos["excess_cagr"],
               "OOS volatility": oos["volatility"], "OOS IR": oos["info_ratio"],
               "OOS trades": int(oos["n_trades"])})
    per_candidate[period] = {label: bt.evaluate(strat, window, p, settings=settings,
                                                score_from=te0)["excess_cagr"]
                             for label, p in cands.items()}

wf_table = pd.DataFrame(wf).drop(columns=["picked", "window", "te0"]).set_index("period")
display(wf_table.style.format({"IS excess": PCT, "OOS excess": PCT, "OOS volatility": "{:.1%}",
                               "OOS IR": "{:+.2f}"}))
is_x, oos_x = wf_table["IS excess"].to_numpy(), wf_table["OOS excess"].to_numpy()
WF_POSITIVE = int((oos_x > 0).sum())
t, p_value = stats.ttest_rel(is_x, oos_x)
print(f"mean IS excess {is_x.mean():+.2%}/yr vs mean OOS excess {oos_x.mean():+.2%}/yr "
      f"(paired t={t:.2f}, p={p_value:.2f}) · {WF_POSITIVE}/{len(oos_x)} periods beat "
      "buy-and-hold out of sample")
print("\nwhat each refit chose (agreement across periods = the knob is load-bearing):")
display(pd.DataFrame([w["picked"] for w in wf], index=wf_table.index).T)
print("\neach CANDIDATE on each test period — IN-SAMPLE for the candidates, so where the edge "
      "sits,\nnever a pass/fail:")
display(pd.DataFrame(per_candidate).T.style.format(PCT)
        .background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3))

### 4b · Ablations on each period's own pick

Each period's pick, trained before its test window, re-scored on that window with one choice
changed: the recency sleeve at 1.0 and switched off, the insider sleeve switched off (§2a shows
it pointing the wrong way on train), and each risk method. A period whose pick already has
`w_insider = 0` scores the same with insider off, and does not count as a win for either. A choice earns its
place out of sample if the pick's own setting beats the alternatives in most periods — for a risk
method, if it is calmer without giving back most of the excess. A `cash_switch` that never fires
in a half-year simply matches `none` there: information, not an error.

In [ ]:
VARIANTS = {
    "as picked": {},
    "recency w=1": {"w_recency": 1.0},
    "recency off": {"w_recency": 0.0},
    "insider off": {"w_insider": 0.0},
    "risk: none": {"risk_method": "none"},
    "risk: cash_switch": {"risk_method": "cash_switch"},
    "risk: vol_target": {"risk_method": "vol_target"},
}
rows = []
for w_ in wf:
    for variant, change in VARIANTS.items():
        st = bt.evaluate(strat, w_["window"], {**w_["picked"], **change}, settings=settings,
                         score_from=w_["te0"])
        rows.append({"period": w_["period"], "variant": variant,
                     "excess": st["excess_cagr"], "volatility": st["volatility"]})
ablations = (pd.DataFrame(rows).pivot(index="period", columns="variant")
             .reindex(columns=pd.MultiIndex.from_product([["excess", "volatility"],
                                                          list(VARIANTS)])))
display(ablations.style.format(PCT))
ex, vol = ablations["excess"], ablations["volatility"]
for sleeve in ("recency", "insider"):
    kept = int((ex["as picked"] > ex[f"{sleeve} off"]).sum())
    print(f"{sleeve} as picked beats {sleeve} off in {kept}/{len(ex)} periods")
for method in ("risk: cash_switch", "risk: vol_target"):
    calmer = int((vol[method] < vol["risk: none"]).sum())
    print(f"{method:18s} calmer than none in {calmer}/{len(vol)} periods · beats buy-and-hold "
          f"in {int((ex[method] > 0).sum())}/{len(ex)}")
print("the refits' own choice of risk method:", [w_["picked"]["risk_method"] for w_ in wf])

## 5 · The untouched holdout — open once

2025+ was never loaded by the optimization notebook. Score the candidates on it **once**, warm,
and do not tune against what you see: a candidate chosen because it won here has turned the
holdout into training data. Annualizing a part year exaggerates in both directions, so each
calendar year is also shown on its own.

### 5a · Against buy-and-hold

In [ ]:
hold, rows = {}, {}
HOLD_YEARS = range(pd.Timestamp(HOLDOUT_START).year, pd.Timestamp(LAST).year + 1)
for label, p in cands.items():
    hold[label] = h = score(p, full, HOLDOUT_START)
    rows[label] = {"CAGR": h["cagr"], "buy & hold": h["bench_cagr"], "excess": h["excess_cagr"],
                   "IR": h["info_ratio"], "volatility": h["volatility"],
                   "B&H volatility": h["bench_volatility"], "beta": h["beta"],
                   "alpha after beta": h["alpha"], "max drawdown": h["max_drawdown"],
                   "trades/yr": h["trades_per_year"]}
    for y in HOLD_YEARS:
        sub = slice_panel(full, d["start"], f"{y}-12-31")
        if len(sub.close.loc[f"{y}-01-01":]) >= 40:
            rows[label][f"excess {y}"] = bt.evaluate(strat, sub, p, settings=settings,
                                                     score_from=f"{y}-01-01")["excess_cagr"]
hold_table = pd.DataFrame(rows).T
display(hold_table.style.format(PCT).format("{:+.2f}", subset=["IR", "beta"])
        .format("{:.1%}", subset=["volatility", "B&H volatility"])
        .format("{:.0f}", subset=["trades/yr"]))

### 5b · Does the ranking still select stocks on the holdout?

The §2a table again, on the holdout (the composite is built on the full history and scored from
the holdout start). Positive on **both** windows = a sleeve carrying real selection; positive in
train and ~0 or negative here is the trap this strategy has fallen into before. A diagnostic
only — choosing weights from it would burn the holdout.

In [ ]:
show_ic({label: sleeve_table(full, p, start=HOLDOUT_START) for label, p in cands.items()},
        "HOLDOUT")

### 5c · The holdout book, name by name — and which rule traded it (FOCUS)

The equity curve, then who was held, when each name was bought and sold, how much of the book it
occupied and what it contributed (`plots.py` beside `strategy.py`), then **why each hold
ended**. That last table has caught this strategy's two worst bugs — displacement owning 122 of
127 exits (v16) and the `min_events` eligibility cliff owning 140 of 196 (v20) — both invisible in
an equity curve. If one mechanism owns most exits, that mechanism *is* the strategy; a high
`score_held` means the name was not sold because its thesis decayed.

In [ ]:
import importlib.util

pf_hold = hold[FOCUS]["pf"]
plots.plot_equity(bt.equity_curve(pf_hold), title=f"Holdout equity — {FOCUS}").show()
_spec = importlib.util.spec_from_file_location("informed_flow_plots", strat.dir / "plots.py")
if_plots = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(if_plots)
if_plots.plot_holdings(holdout, pf_hold).show()

causes = mod["exit_causes"](full, **PARAMS)                 # warm, like the holdout book
causes = causes[causes["exit"] >= pd.Timestamp(HOLDOUT_START, tz="UTC")].reset_index(drop=True)
n = len(causes)
print(f"HOLDOUT ({FOCUS}): {n} completed holds "
      f"({int((bt.trade_records(pf_hold)['status'] == 'Open').sum())} still open)")
if n:
    display(causes.groupby("cause")
            .agg(exits=("cause", "size"), share=("cause", lambda s: len(s) / n),
                 median_bars_held=("bars_held", "median"),
                 median_score_held=("score_held", "median"),
                 still_above_q_in=("score_held", lambda s: (s >= PARAMS["q_in"]).mean()))
            .sort_values("exits", ascending=False)
            .style.format({"share": "{:.0%}", "median_score_held": "{:.3f}",
                           "still_above_q_in": "{:.0%}"}))
    short = causes[causes["bars_held"] <= 5]
    print(f"holds of <= 5 bars: {len(short)}/{n}")
    display(causes.sort_values("bars_held").head(8))

## 6 · Verdict

One row per gate, one column per candidate. A **FAIL** on any hard gate fails the candidate;
**WARN** is a soft gate and **INFO** is reported without a threshold (both set in §0 `GATES`).
The walk-forward gate judges the optimization procedure, so it reads the same for every
candidate; the holdout gates are the ones that judge the candidate itself.

In [ ]:
def gate(ok, value, hard=True):
    """One scoreboard cell: PASS / FAIL (WARN for a soft gate) and the number behind it."""
    return f"{'PASS' if ok else 'FAIL' if hard else 'WARN'} · {value}"


wf_ok = bool(oos_x.mean() > 0) and GATES["wf_min_positive_periods"] <= WF_POSITIVE
pess_min = GATES["pessimistic_min_excess"]
board = {}
for label in cands:
    tr, h, rnd = train[label], hold[label], randoms.loc[label]
    dsr_strict, pess = dsr.loc[label, f"n_eff={N_TRIALS}"], costs.loc[label, "pessimistic fills"]
    cells = {
        "beats buy & hold · train": gate(tr["excess_cagr"] > 0 and tr["info_ratio"] > 0,
                                        f"{tr['excess_cagr']:+.1%} · IR {tr['info_ratio']:+.2f}"),
        "beats buy & hold · holdout": gate(h["excess_cagr"] > 0, f"{h['excess_cagr']:+.1%}"),
        "walk-forward (procedure)": gate(wf_ok, f"{WF_POSITIVE}/{len(oos_x)} · "
                                                f"mean {oos_x.mean():+.1%}"),
        "alpha after beta · holdout": gate(h["alpha"] > 0,
                                          f"{h['alpha']:+.1%} (beta {h['beta']:.2f})"),
        "volatility ≤ buy & hold · holdout": gate(
            h["volatility"] <= h["bench_volatility"],
            f"{h['volatility']:.1%} vs {h['bench_volatility']:.1%}", hard=GATES["vol_is_hard"]),
        "deflated Sharpe": gate(dsr_strict > GATES["dsr_min"], f"{dsr_strict:.2f}"),
        f"beats random p{PCTL}": gate(rnd["real ranking"] > rnd[f"random p{PCTL}"],
                                     f"z {rnd['z']:+.1f}"),
        "pessimistic fills": (f"INFO · {pess:+.1%}" if pess_min is None
                              else gate(pess > pess_min, f"{pess:+.1%}")),
        "no stability spike": gate(stability[label]["spikes"] == 0,
                                   f"{stability[label]['spikes']} spike(s)"),
    }
    failed = [k for k, v in cells.items() if v.startswith("FAIL")]
    cells["VERDICT"] = "PASS" if not failed else f"FAIL ({len(failed)} gate(s))"
    board[label] = cells

scoreboard = pd.DataFrame(board)
COLOR = {"PASS": "#d8efd6", "FAIL": "#f6d5d1", "WARN": "#fbe8c6", "INFO": ""}
display(scoreboard.style.map(lambda v: f"background: {COLOR.get(v.split(' ')[0], '')}"))

## 7 · Promote a candidate into `config.yaml`

Off by default, and deliberately so: this **rewrites a tracked file**, and what ships is a
decision, not a side effect of running a notebook. It refuses a candidate whose verdict failed
unless `FORCE = True` (say why in the commit). The rewrite is surgical — only the `value:` lines
change, every comment survives — and it verifies itself by reloading the strategy, restoring the
original file if the config no longer loads or a value does not round-trip.

In [ ]:
import pathlib
import re

PROMOTE = FOCUS        # which candidate to write into config.yaml
WRITE_BACK = False     # True rewrites strategies/informed_flow/config.yaml
FORCE = False          # True writes a candidate that failed the verdict


def _yaml_scalar(v):
    """Render one value the way config.yaml writes it."""
    if isinstance(v, bool):
        return "true" if v else "false"
    return f'"{v}"' if isinstance(v, str) else repr(v)


def promote(params, *, apply=False):
    """Rewrite `params.<key>.value` for every knob that differs; verify by reloading."""
    path = pathlib.Path(strat.dir) / "config.yaml"
    original = path.read_text()
    cfg = yaml.safe_load(original)
    lines, changes = original.splitlines(), []
    for key, new in params.items():
        spec = cfg["params"].get(key)
        if not isinstance(spec, dict) or spec.get("value") == new:
            continue
        at = next((i for i, ln in enumerate(lines) if re.fullmatch(rf"  {key}:\s*", ln)), None)
        if at is None:
            print(f"  ! {key}: no long-form entry to write to — edit it by hand")
            continue
        end = next((i for i in range(at + 1, len(lines))
                    if lines[i][:3].strip() and not lines[i].startswith("    ")), len(lines))
        for i in range(at + 1, end):
            if lines[i].split("#")[0].strip().startswith("value:"):
                note = lines[i].partition("#")[2]
                lines[i] = f"    value: {_yaml_scalar(new)}" + (f"   #{note}" if note else "")
        changes.append((key, spec.get("value"), new))
    for key, old, new in changes:
        print(f"  {key:18s} {old!r:>12} -> {new!r}")
    if not changes:
        print("config.yaml already ships these values — nothing to promote.")
        return changes
    if not apply:
        print(f"\n{len(changes)} knob(s) would change. Set WRITE_BACK = True to apply.")
        return changes
    path.write_text("\n".join(lines) + "\n")
    try:
        reloaded = strategies.load(strat.name)            # the config invariants run here
    except Exception:
        path.write_text(original)
        print("REVERTED — the rewritten config does not load:")
        raise
    if bad := {k: v for k, v in params.items() if reloaded.params.get(k) != v}:
        path.write_text(original)
        raise AssertionError(f"REVERTED — values did not round-trip: {bad}")
    print(f"\nwrote {path} — commit the diff, then re-run both notebooks from the top.")
    return changes


verdict = scoreboard.loc["VERDICT", PROMOTE]
print(f"{PROMOTE}: verdict {verdict}\n")
if WRITE_BACK and verdict != "PASS" and not FORCE:
    raise RuntimeError(f"{PROMOTE} failed the verdict — set FORCE = True to write it anyway")
_ = promote(cands[PROMOTE], apply=WRITE_BACK)

## Read the result honestly

- The **holdout** and **walk-forward** numbers are the real ones; §3 can only show fragility.
- The walk-forward judges the **procedure**; only the holdout judges a **candidate**. A candidate
  needs both.
- Comparing many candidates on the holdout and shipping the winner is selection on the holdout:
  pick among candidates on §3 and the walk-forward, then read the holdout as confirmation.
- ~7 years of train is a short sample: prefer consistency across years (§3b) to a high headline.
- A **risk overlay** earns its complexity only if §4b shows it calmer out of sample without
  handing back most of the excess.
- The yfinance ratings sleeve is survivor-only; insider (SEC) and prices are not. Keep claims to
  this liquid universe.